In [3]:
import itertools
import time
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [4]:
DATASET_NAME = "CLAP 3.0s"

DATASET_PATH = Path(
    "/Users/bhavaykhatri/Desktop/msclap_2023/"
    "singBAP_dataset_clap-2023_3.0s.parquet"
)

OUTPUT_DIR = Path(
    "clap_3_0s_feature_selection_results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET_EXPERIENCE = [
    "intermediate",
    "professional",
]

TARGET_CLASSES = [
    "correct",
    "arched_back",
    "hunched_back",
    "sideways",
    "chest_breathing",
    "over_articulation",
    "under_articulation",
]

OUTER_RANDOM_STATE = 42
INNER_RANDOM_STATE = 43

BASELINE_MACRO_F1 = 0.5301

In [5]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )

df = pd.read_parquet(
    DATASET_PATH
)

print("Original shape:", df.shape)

df = df[
    df["experience"].isin(
        TARGET_EXPERIENCE
    )
].copy()

df = df[
    df["condition"].isin(
        TARGET_CLASSES
    )
].copy()

df = df.reset_index(
    drop=True
)

print("Filtered shape:", df.shape)

print("\nCondition distribution:")
print(
    df["condition"]
    .value_counts()
    .reindex(TARGET_CLASSES)
)

print("\nExperience distribution:")
print(
    df["experience"].value_counts()
)

Original shape: (4116, 13)
Filtered shape: (3438, 13)

Condition distribution:
condition
correct               638
arched_back           405
hunched_back          556
sideways              521
chest_breathing       512
over_articulation     411
under_articulation    395
Name: count, dtype: int64

Experience distribution:
experience
intermediate    2794
professional     644
Name: count, dtype: int64


In [6]:
def decode_embedding(
    value,
    dtype=np.float32,
):
    if isinstance(
        value,
        (bytes, bytearray, memoryview),
    ):
        return np.frombuffer(
            value,
            dtype=dtype,
        ).copy()

    return np.asarray(
        value,
        dtype=dtype,
    ).reshape(-1)


decoded_embeddings = [
    decode_embedding(value)
    for value in df["embedding"]
]

embedding_dimensions = {
    embedding.shape[0]
    for embedding in decoded_embeddings
}

if len(embedding_dimensions) != 1:
    raise ValueError(
        "Inconsistent embedding dimensions: "
        f"{sorted(embedding_dimensions)}"
    )

X = np.vstack(
    decoded_embeddings
).astype(
    np.float32,
    copy=False,
)

y = (
    df["condition"]
    .astype(str)
    .to_numpy()
)

groups = (
    df["filename"]
    .astype(str)
    .to_numpy()
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print(
    "Unique recordings:",
    len(np.unique(groups)),
)
print(
    "Number of classes:",
    len(np.unique(y)),
)

X shape: (3438, 1024)
y shape: (3438,)
Unique recordings: 3046
Number of classes: 7


In [7]:
finite_rows = np.isfinite(
    X
).all(axis=1)

removed_rows = (
    len(finite_rows)
    - finite_rows.sum()
)

print(
    "Rows containing NaN or infinity:",
    removed_rows,
)

if removed_rows > 0:
    X = X[finite_rows]
    y = y[finite_rows]
    groups = groups[finite_rows]

    df = (
        df.loc[finite_rows]
        .reset_index(drop=True)
    )

print("Final X shape:", X.shape)

Rows containing NaN or infinity: 0
Final X shape: (3438, 1024)


In [8]:
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=OUTER_RANDOM_STATE,
)

train_idx, test_idx = next(
    outer_splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

train_groups = groups[train_idx]
test_groups = groups[test_idx]

shared_outer_recordings = (
    set(train_groups)
    & set(test_groups)
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print(
    "Shared recordings:",
    len(shared_outer_recordings),
)

Train: (2750, 1024)
Test: (688, 1024)
Shared recordings: 0


In [9]:
inner_splitter = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=INNER_RANDOM_STATE,
)

feature_train_idx, validation_idx = next(
    inner_splitter.split(
        X_train,
        y_train,
        groups=train_groups,
    )
)

X_feature_train = X_train[
    feature_train_idx
]

y_feature_train = y_train[
    feature_train_idx
]

X_validation = X_train[
    validation_idx
]

y_validation = y_train[
    validation_idx
]

feature_train_groups = train_groups[
    feature_train_idx
]

validation_groups = train_groups[
    validation_idx
]

shared_inner_recordings = (
    set(feature_train_groups)
    & set(validation_groups)
)

print(
    "Feature-selection training:",
    X_feature_train.shape,
)

print(
    "Validation:",
    X_validation.shape,
)

print(
    "Shared inner recordings:",
    len(shared_inner_recordings),
)

Feature-selection training: (2062, 1024)
Validation: (688, 1024)
Shared inner recordings: 0


In [10]:
SEARCH_MODEL = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        class_weight="balanced",
        dual=False,
        tol=1e-3,
        max_iter=50000,
        random_state=42,
    ),
)

score_cache = {}


def evaluate_feature_indices(
    feature_indices,
):
    feature_indices = np.asarray(
        feature_indices,
        dtype=int,
    )

    if feature_indices.size == 0:
        return np.nan

    cache_key = tuple(
        sorted(feature_indices.tolist())
    )

    if cache_key in score_cache:
        return score_cache[cache_key]

    model = clone(
        SEARCH_MODEL
    )

    model.fit(
        X_feature_train[
            :,
            feature_indices,
        ],
        y_feature_train,
    )

    predictions = model.predict(
        X_validation[
            :,
            feature_indices,
        ]
    )

    score = f1_score(
        y_validation,
        predictions,
        average="macro",
        zero_division=0,
    )

    score_cache[cache_key] = score

    return score

In [11]:
selection_results = []
selected_feature_sets = {}


def record_selection(
    method,
    family,
    feature_indices,
    validation_macro_f1,
    elapsed_time,
):
    feature_indices = np.asarray(
        feature_indices,
        dtype=int,
    )

    selected_feature_sets[
        method
    ] = feature_indices.copy()

    selection_results.append({
        "Method": method,
        "Family": family,
        "Selected Features": len(
            feature_indices
        ),
        "Validation Macro F1": (
            validation_macro_f1
        ),
        "Selection Time (s)": (
            elapsed_time
        ),
    })

    print(
        f"{method}: "
        f"{len(feature_indices)} features, "
        f"validation Macro F1="
        f"{validation_macro_f1:.4f}, "
        f"time={elapsed_time:.2f}s"
    )

In [12]:
all_feature_indices = np.arange(
    X_feature_train.shape[1]
)

start_time = time.time()

all_features_validation_f1 = (
    evaluate_feature_indices(
        all_feature_indices
    )
)

record_selection(
    method="All Features",
    family="Baseline",
    feature_indices=all_feature_indices,
    validation_macro_f1=(
        all_features_validation_f1
    ),
    elapsed_time=(
        time.time() - start_time
    ),
)

All Features: 1024 features, validation Macro F1=0.3981, time=13.40s


In [13]:
ANOVA_K_VALUES = [
    128,
    256,
    384,
    512,
    640,
    768,
    832,
    896,
    960,
    992,
]

for k in ANOVA_K_VALUES:
    start_time = time.time()

    selector = SelectKBest(
        score_func=f_classif,
        k=k,
    )

    selector.fit(
        X_feature_train,
        y_feature_train,
    )

    selected_indices = (
        selector.get_support(
            indices=True
        )
    )

    validation_f1 = (
        evaluate_feature_indices(
            selected_indices
        )
    )

    record_selection(
        method=f"ANOVA K={k}",
        family="ANOVA",
        feature_indices=selected_indices,
        validation_macro_f1=validation_f1,
        elapsed_time=(
            time.time() - start_time
        ),
    )

ANOVA K=128: 128 features, validation Macro F1=0.3117, time=0.40s
ANOVA K=256: 256 features, validation Macro F1=0.3414, time=1.41s
ANOVA K=384: 384 features, validation Macro F1=0.3967, time=2.69s
ANOVA K=512: 512 features, validation Macro F1=0.3848, time=4.12s
ANOVA K=640: 640 features, validation Macro F1=0.3862, time=6.66s
ANOVA K=768: 768 features, validation Macro F1=0.4128, time=7.29s
ANOVA K=832: 832 features, validation Macro F1=0.4080, time=8.77s
ANOVA K=896: 896 features, validation Macro F1=0.4113, time=13.20s
ANOVA K=960: 960 features, validation Macro F1=0.3976, time=12.38s
ANOVA K=992: 992 features, validation Macro F1=0.4042, time=14.28s


In [14]:
L1_C_VALUES = [
    0.001,
    0.003,
    0.01,
    0.03,
    0.1,
]

l1_scaler = StandardScaler()

X_feature_train_l1 = (
    l1_scaler.fit_transform(
        X_feature_train
    )
)

for c_value in L1_C_VALUES:
    start_time = time.time()

    l1_model = LinearSVC(
        C=c_value,
        penalty="l1",
        dual=False,
        class_weight="balanced",
        tol=1e-3,
        max_iter=50000,
        random_state=42,
    )

    l1_model.fit(
        X_feature_train_l1,
        y_feature_train,
    )

    selected_mask = np.any(
        np.abs(
            l1_model.coef_
        ) > 1e-8,
        axis=0,
    )

    selected_indices = np.flatnonzero(
        selected_mask
    )

    if len(selected_indices) == 0:
        print(
            f"L1-SVM C={c_value}: "
            "zero features selected."
        )
        continue

    validation_f1 = (
        evaluate_feature_indices(
            selected_indices
        )
    )

    record_selection(
        method=f"L1-SVM C={c_value}",
        family="L1-SVM",
        feature_indices=selected_indices,
        validation_macro_f1=validation_f1,
        elapsed_time=(
            time.time() - start_time
        ),
    )

L1-SVM C=0.001: zero features selected.
L1-SVM C=0.003: 18 features, validation Macro F1=0.2170, time=0.25s
L1-SVM C=0.01: 171 features, validation Macro F1=0.3171, time=1.09s
L1-SVM C=0.03: 407 features, validation Macro F1=0.3857, time=8.05s
L1-SVM C=0.1: 702 features, validation Macro F1=0.3812, time=13.82s


In [15]:
def sequential_forward_selection(
    candidate_indices,
    maximum_selected=15,
):
    candidate_indices = list(
        map(int, candidate_indices)
    )

    selected = []
    remaining = candidate_indices.copy()

    best_overall_score = -np.inf
    best_overall_subset = None

    history = []

    maximum_steps = min(
        maximum_selected,
        len(candidate_indices),
    )

    for step in range(maximum_steps):
        best_step_feature = None
        best_step_score = -np.inf

        for feature_index in remaining:
            trial_subset = (
                selected
                + [feature_index]
            )

            score = evaluate_feature_indices(
                trial_subset
            )

            if score > best_step_score:
                best_step_score = score
                best_step_feature = (
                    feature_index
                )

        selected.append(
            best_step_feature
        )

        remaining.remove(
            best_step_feature
        )

        history.append({
            "Step": step + 1,
            "Added Feature": (
                best_step_feature
            ),
            "Selected Features": len(
                selected
            ),
            "Validation Macro F1": (
                best_step_score
            ),
        })

        print(
            f"Step {step + 1}: "
            f"added feature "
            f"{best_step_feature}, "
            f"Macro F1="
            f"{best_step_score:.4f}"
        )

        if best_step_score > best_overall_score:
            best_overall_score = (
                best_step_score
            )

            best_overall_subset = (
                selected.copy()
            )

    return (
        np.asarray(
            best_overall_subset,
            dtype=int,
        ),
        best_overall_score,
        pd.DataFrame(history),
    )

In [16]:
SFS_PREFILTER_K = 30
SFS_MAXIMUM_SELECTED = 15

sfs_prefilter = SelectKBest(
    score_func=f_classif,
    k=SFS_PREFILTER_K,
)

sfs_prefilter.fit(
    X_feature_train,
    y_feature_train,
)

sfs_candidate_indices = (
    sfs_prefilter.get_support(
        indices=True
    )
)

print(
    "SFS candidate features:",
    sfs_candidate_indices,
)

start_time = time.time()

(
    sequential_indices,
    sequential_validation_f1,
    sequential_history_df,
) = sequential_forward_selection(
    candidate_indices=(
        sfs_candidate_indices
    ),
    maximum_selected=(
        SFS_MAXIMUM_SELECTED
    ),
)

record_selection(
    method="Sequential Forward",
    family="Sequential",
    feature_indices=sequential_indices,
    validation_macro_f1=(
        sequential_validation_f1
    ),
    elapsed_time=(
        time.time() - start_time
    ),
)

sequential_history_df

SFS candidate features: [  58   95   96  116  125  131  133  230  239  322  388  394  490  496
  518  612  640  748  750  759  772  780  814  864  867  886  893  924
  930 1007]
Step 1: added feature 322, Macro F1=0.1068
Step 2: added feature 388, Macro F1=0.1566
Step 3: added feature 518, Macro F1=0.1673
Step 4: added feature 1007, Macro F1=0.1823
Step 5: added feature 58, Macro F1=0.2000
Step 6: added feature 239, Macro F1=0.2096
Step 7: added feature 772, Macro F1=0.2138
Step 8: added feature 125, Macro F1=0.2109
Step 9: added feature 780, Macro F1=0.2152
Step 10: added feature 131, Macro F1=0.2217
Step 11: added feature 394, Macro F1=0.2261
Step 12: added feature 886, Macro F1=0.2303
Step 13: added feature 95, Macro F1=0.2272
Step 14: added feature 490, Macro F1=0.2277
Step 15: added feature 924, Macro F1=0.2319
Sequential Forward: 15 features, validation Macro F1=0.2319, time=7.14s


,Step,Added Feature,Selected Features,Validation Macro F1
0,1,322,1,0.106818
1,2,388,2,0.156615
2,3,518,3,0.167253
3,4,1007,4,0.182258
4,5,58,5,0.199957
5,6,239,6,0.209562
6,7,772,7,0.213765
7,8,125,8,0.210899
8,9,780,9,0.215176
9,10,131,10,0.221668


In [17]:
def exhaustive_feature_selection(
    candidate_indices,
    minimum_subset_size=3,
    maximum_subset_size=4,
):
    candidate_indices = list(
        map(int, candidate_indices)
    )

    best_score = -np.inf
    best_subset = None
    evaluated_subsets = 0

    history = []

    for subset_size in range(
        minimum_subset_size,
        maximum_subset_size + 1,
    ):
        print(
            f"Testing every subset "
            f"of size {subset_size}..."
        )

        for subset in itertools.combinations(
            candidate_indices,
            subset_size,
        ):
            score = evaluate_feature_indices(
                subset
            )

            evaluated_subsets += 1

            if score > best_score:
                best_score = score
                best_subset = subset

                history.append({
                    "Evaluated Subsets": (
                        evaluated_subsets
                    ),
                    "Subset Size": (
                        subset_size
                    ),
                    "Validation Macro F1": (
                        score
                    ),
                    "Feature Indices": (
                        list(subset)
                    ),
                })

                print(
                    f"New best: "
                    f"F1={score:.4f}, "
                    f"features={subset}"
                )

    return (
        np.asarray(
            best_subset,
            dtype=int,
        ),
        best_score,
        evaluated_subsets,
        pd.DataFrame(history),
    )

In [18]:
BRUTE_FORCE_TOP_K = 8
BRUTE_FORCE_MINIMUM_SIZE = 3
BRUTE_FORCE_MAXIMUM_SIZE = 4

brute_prefilter = SelectKBest(
    score_func=f_classif,
    k=BRUTE_FORCE_TOP_K,
)

brute_prefilter.fit(
    X_feature_train,
    y_feature_train,
)

brute_candidate_indices = (
    brute_prefilter.get_support(
        indices=True
    )
)

print(
    "Brute-force candidates:",
    brute_candidate_indices,
)

start_time = time.time()

(
    brute_force_indices,
    brute_force_validation_f1,
    evaluated_subsets,
    brute_force_history_df,
) = exhaustive_feature_selection(
    candidate_indices=(
        brute_candidate_indices
    ),
    minimum_subset_size=(
        BRUTE_FORCE_MINIMUM_SIZE
    ),
    maximum_subset_size=(
        BRUTE_FORCE_MAXIMUM_SIZE
    ),
)

record_selection(
    method="Brute Force",
    family="Exhaustive",
    feature_indices=(
        brute_force_indices
    ),
    validation_macro_f1=(
        brute_force_validation_f1
    ),
    elapsed_time=(
        time.time() - start_time
    ),
)

print(
    "Subsets evaluated:",
    evaluated_subsets,
)

brute_force_history_df

Brute-force candidates: [  95   96  131  748  814  886  893 1007]
Testing every subset of size 3...
New best: F1=0.1336, features=(95, 96, 131)
New best: F1=0.1341, features=(95, 96, 748)
New best: F1=0.1704, features=(95, 96, 814)
New best: F1=0.1712, features=(96, 131, 814)
New best: F1=0.1916, features=(814, 886, 1007)
Testing every subset of size 4...
New best: F1=0.1962, features=(96, 131, 814, 1007)
Brute Force: 4 features, validation Macro F1=0.1962, time=1.15s
Subsets evaluated: 126


,Evaluated Subsets,Subset Size,Validation Macro F1,Feature Indices
0,1,3,0.133565,"[95, 96, 131]"
1,2,3,0.134068,"[95, 96, 748]"
2,3,3,0.170359,"[95, 96, 814]"
3,23,3,0.171215,"[96, 131, 814]"
4,54,3,0.191647,"[814, 886, 1007]"
5,98,4,0.196200,"[96, 131, 814, 1007]"


In [19]:
selection_results_df = pd.DataFrame(
    selection_results
)

selection_results_df = (
    selection_results_df
    .sort_values(
        "Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

selection_results_df

,Method,Family,Selected Features,Validation Macro F1,Selection Time (s)
0,ANOVA K=768,ANOVA,768,0.412803,7.290584
1,ANOVA K=896,ANOVA,896,0.411285,13.204863
2,ANOVA K=832,ANOVA,832,0.408036,8.770892
3,ANOVA K=992,ANOVA,992,0.404155,14.283950
4,All Features,Baseline,1024,0.398079,13.404099
5,ANOVA K=960,ANOVA,960,0.397581,12.382644
6,ANOVA K=384,ANOVA,384,0.396704,2.694564
7,ANOVA K=640,ANOVA,640,0.386161,6.659454
8,L1-SVM C=0.03,L1-SVM,407,0.385743,8.054954
9,ANOVA K=512,ANOVA,512,0.384761,4.123829


In [20]:
best_family_indices = (
    selection_results_df
    .groupby("Family")[
        "Validation Macro F1"
    ]
    .idxmax()
)

best_family_rows = (
    selection_results_df
    .loc[best_family_indices]
    .sort_values(
        "Validation Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

best_family_rows

,Method,Family,Selected Features,Validation Macro F1,Selection Time (s)
0,ANOVA K=768,ANOVA,768,0.412803,7.290584
1,All Features,Baseline,1024,0.398079,13.404099
2,L1-SVM C=0.03,L1-SVM,407,0.385743,8.054954
3,Sequential Forward,Sequential,15,0.231910,7.141119
4,Brute Force,Exhaustive,4,0.196200,1.147028


In [21]:
FINAL_FEATURE_SETS = {}

for _, row in best_family_rows.iterrows():
    method_name = row["Method"]

    if method_name == "All Features":
        final_indices = np.arange(
            X_train.shape[1]
        )

    elif method_name.startswith(
        "ANOVA K="
    ):
        k = int(
            method_name.split("=")[1]
        )

        final_selector = SelectKBest(
            score_func=f_classif,
            k=k,
        )

        final_selector.fit(
            X_train,
            y_train,
        )

        final_indices = (
            final_selector.get_support(
                indices=True
            )
        )

    elif method_name.startswith(
        "L1-SVM C="
    ):
        c_value = float(
            method_name.split("=")[1]
        )

        final_l1_scaler = (
            StandardScaler()
        )

        X_train_l1 = (
            final_l1_scaler
            .fit_transform(X_train)
        )

        final_l1_model = LinearSVC(
            C=c_value,
            penalty="l1",
            dual=False,
            class_weight="balanced",
            tol=1e-3,
            max_iter=50000,
            random_state=42,
        )

        final_l1_model.fit(
            X_train_l1,
            y_train,
        )

        final_mask = np.any(
            np.abs(
                final_l1_model.coef_
            ) > 1e-8,
            axis=0,
        )

        final_indices = np.flatnonzero(
            final_mask
        )

    else:
        final_indices = (
            selected_feature_sets[
                method_name
            ]
        )

    FINAL_FEATURE_SETS[
        method_name
    ] = np.asarray(
        final_indices,
        dtype=int,
    )

    print(
        method_name,
        "->",
        len(final_indices),
        "features",
    )

ANOVA K=768 -> 768 features
All Features -> 1024 features
L1-SVM C=0.03 -> 447 features
Sequential Forward -> 15 features
Brute Force -> 4 features


In [22]:
MODELS = {
    "MLP": make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(
                256,
                128,
            ),
            early_stopping=True,
            max_iter=300,
            random_state=42,
        ),
    ),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(
            n_neighbors=15,
            metric="cosine",
            n_jobs=-1,
        ),
    ),

    "Random Forest": (
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
    ),

    "Linear SVM": make_pipeline(
        StandardScaler(),
        LinearSVC(
            class_weight="balanced",
            max_iter=10000,
            random_state=42,
        ),
    ),
}

In [23]:
final_results = []

for method_name, feature_indices in (
    FINAL_FEATURE_SETS.items()
):
    print("\n" + "=" * 70)

    print(
        f"{method_name}: "
        f"{len(feature_indices)} features"
    )

    X_train_selected = X_train[
        :,
        feature_indices,
    ]

    X_test_selected = X_test[
        :,
        feature_indices,
    ]

    for model_name, base_model in (
        MODELS.items()
    ):
        print(
            f"Training {model_name}..."
        )

        model = clone(
            base_model
        )

        start_time = time.time()

        model.fit(
            X_train_selected,
            y_train,
        )

        predictions = model.predict(
            X_test_selected
        )

        elapsed_time = (
            time.time() - start_time
        )

        final_results.append({
            "Embedding": DATASET_NAME,
            "Feature Method": (
                method_name
            ),
            "Selected Features": len(
                feature_indices
            ),
            "Model": model_name,
            "Accuracy": accuracy_score(
                y_test,
                predictions,
            ),
            "Balanced Accuracy": (
                balanced_accuracy_score(
                    y_test,
                    predictions,
                )
            ),
            "Macro F1": f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0,
            ),
            "Train/Eval Time (s)": (
                elapsed_time
            ),
        })


ANOVA K=768: 768 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

All Features: 1024 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

L1-SVM C=0.03: 447 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

Sequential Forward: 15 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...

Brute Force: 4 features
Training MLP...
Training KNN...
Training Random Forest...
Training Linear SVM...


In [24]:
final_results_df = pd.DataFrame(
    final_results
)

final_results_df = (
    final_results_df
    .sort_values(
        "Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

final_results_df

,Embedding,Feature Method,Selected Features,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time (s)
0,CLAP 3.0s,All Features,1024,MLP,0.524709,0.520422,0.530136,24.429268
1,CLAP 3.0s,L1-SVM C=0.03,447,MLP,0.530523,0.526659,0.528095,16.404913
2,CLAP 3.0s,ANOVA K=768,768,MLP,0.518895,0.524111,0.527157,24.201756
3,CLAP 3.0s,All Features,1024,Linear SVM,0.469477,0.470717,0.471419,33.989480
4,CLAP 3.0s,ANOVA K=768,768,Random Forest,0.470930,0.479065,0.471356,2.948761
5,CLAP 3.0s,All Features,1024,Random Forest,0.456395,0.463107,0.458579,5.521292
6,CLAP 3.0s,ANOVA K=768,768,Linear SVM,0.454942,0.457430,0.456018,21.973278
7,CLAP 3.0s,L1-SVM C=0.03,447,Random Forest,0.452035,0.459515,0.454561,2.235425
8,CLAP 3.0s,L1-SVM C=0.03,447,Linear SVM,0.422965,0.426654,0.423547,5.795733
9,CLAP 3.0s,Sequential Forward,15,MLP,0.343023,0.346036,0.334039,14.276573


In [25]:
best_method_indices = (
    final_results_df
    .groupby("Feature Method")[
        "Macro F1"
    ]
    .idxmax()
)

best_result_per_method = (
    final_results_df
    .loc[best_method_indices]
    .sort_values(
        "Macro F1",
        ascending=False,
    )
    .reset_index(drop=True)
)

best_result_per_method[
    [
        "Feature Method",
        "Selected Features",
        "Model",
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
        "Train/Eval Time (s)",
    ]
]

,Feature Method,Selected Features,Model,Accuracy,Balanced Accuracy,Macro F1,Train/Eval Time (s)
0,All Features,1024,MLP,0.524709,0.520422,0.530136,24.429268
1,L1-SVM C=0.03,447,MLP,0.530523,0.526659,0.528095,16.404913
2,ANOVA K=768,768,MLP,0.518895,0.524111,0.527157,24.201756
3,Sequential Forward,15,MLP,0.343023,0.346036,0.334039,14.276573
4,Brute Force,4,KNN,0.225291,0.220774,0.217059,0.158778


In [26]:
comparison_df = (
    best_result_per_method.copy()
)

comparison_df[
    "Macro F1 Change"
] = (
    comparison_df["Macro F1"]
    - BASELINE_MACRO_F1
)

comparison_df[
    "Feature Reduction (%)"
] = (
    1
    - (
        comparison_df[
            "Selected Features"
        ]
        / X.shape[1]
    )
) * 100

comparison_df[
    [
        "Feature Method",
        "Selected Features",
        "Model",
        "Macro F1",
        "Macro F1 Change",
        "Feature Reduction (%)",
    ]
]

,Feature Method,Selected Features,Model,Macro F1,Macro F1 Change,Feature Reduction (%)
0,All Features,1024,MLP,0.530136,0.000036,0.000000
1,L1-SVM C=0.03,447,MLP,0.528095,-0.002005,56.347656
2,ANOVA K=768,768,MLP,0.527157,-0.002943,25.000000
3,Sequential Forward,15,MLP,0.334039,-0.196061,98.535156
4,Brute Force,4,KNN,0.217059,-0.313041,99.609375


In [27]:
selection_results_df.to_csv(
    OUTPUT_DIR
    / "validation_feature_selection.csv",
    index=False,
)

final_results_df.to_csv(
    OUTPUT_DIR
    / "heldout_test_results.csv",
    index=False,
)

best_result_per_method.to_csv(
    OUTPUT_DIR
    / "best_result_per_method.csv",
    index=False,
)

comparison_df.to_csv(
    OUTPUT_DIR
    / "baseline_comparison.csv",
    index=False,
)

sequential_history_df.to_csv(
    OUTPUT_DIR
    / "sequential_history.csv",
    index=False,
)

brute_force_history_df.to_csv(
    OUTPUT_DIR
    / "brute_force_history.csv",
    index=False,
)

np.savez(
    OUTPUT_DIR
    / "selected_feature_indices.npz",
    **{
        method_name
        .lower()
        .replace(" ", "_")
        .replace("=", "_")
        .replace(".", "_"): indices

        for method_name, indices
        in FINAL_FEATURE_SETS.items()
    },
)

print(
    "Saved results to:",
    OUTPUT_DIR.resolve(),
)

Saved results to: /Users/bhavaykhatri/Desktop/Assignments/audio_data_benchmarking_mml_lab/clap_3_0s_feature_selection_results
